In [20]:
import openai, json

client = openai.OpenAI()
messages = []

In [21]:
import requests
import json

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

def get_popular_movies():
    """인기 영화 목록을 반환합니다."""
    resp = requests.get(f"{BASE_URL}/movies")
    resp.raise_for_status()
    return json.dumps(resp.json(), ensure_ascii=False)

def get_movie_details(id):
    """ID로 영화 상세 정보를 조회합니다."""
    resp = requests.get(f"{BASE_URL}/movies/{id}")
    resp.raise_for_status()
    return json.dumps(resp.json(), ensure_ascii=False)

def get_movie_credits(id):
    """영화의 출연진 및 제작진 정보를 반환합니다."""
    resp = requests.get(f"{BASE_URL}/movies/{id}/credits")
    resp.raise_for_status()
    return json.dumps(resp.json(), ensure_ascii=False)

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits
}

In [22]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of popular movies."
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get the details of a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The ID of the movie."
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get the credits of a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The ID of the movie."
                    }
                },
                "required": ["id"]
            }
        }
    }
]

In [23]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role":"assistant", 
                "content":message.content or "",
                "tool_calls":[
                    {
                        "id":tool_call.id,
                        "type":"function",
                        "function":{
                            "name":tool_call.function.name,
                            "arguments":tool_call.function.arguments
                        },
                    } for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)
            
            print(f"Ran {function_name} with args {arguments} for a result of {result}")

            messages.append({
                "role":"tool",
                "tool_call_id":tool_call.id,
                "name":function_name,
                "content":result,
            })
        Call_ai()
    else :
        messages.append({"role":"assistant", "content":message.content})
        print(f"AI : {message.content}")



def Call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [24]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role":"user", "content":message})
        print(f"User : {message}")
        Call_ai()

User : 안녕 난 영군이야
AI : 안녕하세요, 영군님! 어떻게 도와드릴까요?
User : 넌 무엇을 할 수 있어?
AI : 저는 다양한 정보를 제공하고, 질문에 답하며, 여러 주제에 대해 대화할 수 있습니다. 영화, 책, 과학, 역사 등 다양한 분야에 대한 정보를 찾거나 추천할 수도 있습니다. 또 필요한 경우 특정 영화의 세부 사항이나 출연진 정보를 조회할 수도 있습니다. 어떤 도움을 드릴까요?
User : 인기 영화 목록 알려줘
Calling function: get_popular_movies with {}
Ran get_popular_movies with args {} for a result of [{"adult": false, "backdrop_path": "https://image.tmdb.org/t/p/w1280/7HKpc11uQfxnw0Y8tRUYn1fsKqE.jpg", "genre_ids": [878, 28, 53], "id": 1236153, "original_language": "en", "original_title": "Mercy", "overview": "In the near future, a detective stands on trial accused of murdering his wife. He has ninety minutes to prove his innocence to the advanced AI Judge he once championed, before it determines his fate.", "popularity": 649.4032, "poster_path": "https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg", "release_date": "2026-01-20", "title": "Mercy", "video": false, "vote_average": 7.054, "vote_count": 468}, {"adult": false, "backdrop_pa